In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler


df = pd.read_csv("sample_trading_data.csv")


df['open_time'] = pd.to_datetime(df['open_time'])
df['close_time'] = pd.to_datetime(df['close_time'])

df = df.sort_values(by='open_time').reset_index(drop=True)


df['time_since_last_trade'] = (
    df['open_time'] - df['close_time'].shift(1)
).dt.total_seconds()


df['hour_of_day'] = df['open_time'].dt.hour

df['day_of_week'] = df['open_time'].dt.dayofweek

df['trade_duration_seconds'] = (
    df['close_time'] - df['open_time']
).dt.total_seconds()


In [5]:
def get_session(hour):
    if 7 <= hour < 16:
        return 1   # London
    elif 13 <= hour < 22:
        return 2   # New York
    elif 0 <= hour < 9:
        return 3   # Asian
    else:
        return 0   # Off Hours

df['session'] = df['hour_of_day'].apply(get_session)



In [6]:
baseline_lot = df['lot_size'].mean()


df['lot_size_vs_baseline'] = (
    df['lot_size'] / baseline_lot
)

df['lot_size_change_from_last'] = (
    df['lot_size'].diff()
)



In [7]:
df['position_size_percentile'] = (
    df['lot_size'].rank(pct=True)
)


df['account_balance'] = 10000 + df['pnl'].cumsum()

df['risk_percent_of_balance'] = (
    abs(df['entry_price'] - df['stop_loss']) *
    df['lot_size'] /
    (df['account_balance'] + 1e-6)
)


In [8]:
losses = (df['pnl'] < 0).astype(int)

consecutive_losses = []
count = 0

for val in losses:
    if val == 1:
        count += 1
    else:
        count = 0
    consecutive_losses.append(count)

df['consecutive_losses'] = consecutive_losses



In [9]:
df['trades_in_last_30_min'] = 0

for i in range(len(df)):
    current_time = df.loc[i, 'open_time']
    
    past_trades = df[
        (df['open_time'] >= current_time - pd.Timedelta(minutes=30)) &
        (df['open_time'] < current_time)
    ]
    
    df.loc[i, 'trades_in_last_30_min'] = len(past_trades)


df['trades_in_session'] = (
    df.groupby('session').cumcount() + 1
)


In [10]:
same_symbol = []
count = 1

for i in range(len(df)):
    if i == 0:
        same_symbol.append(1)
    else:
        if df.loc[i, 'pair'] == df.loc[i-1, 'pair']:
            count += 1
        else:
            count = 1
        
        same_symbol.append(count)

df['same_symbol_consecutive'] = same_symbol



In [11]:

df['pnl_last_trade'] = df['pnl'].shift(1)


df['cumulative_pnl_session'] = (
    df.groupby('session')['pnl'].cumsum()
)


df['pnl_volatility'] = (
    df['pnl'].rolling(window=5).std()
)


df['peak_balance'] = df['account_balance'].cummax()

df['drawdown_from_peak'] = (
    df['peak_balance'] - df['account_balance']
)

In [12]:
df['sl_present'] = (
    df['stop_loss'].notnull()
).astype(int)



df['risk_reward_ratio'] = (
    abs(df['take_profit'] - df['entry_price']) /
    (abs(df['entry_price'] - df['stop_loss']) + 1e-6)
)


df['emotional_momentum_score'] = (
    df['consecutive_losses'] * 0.4 +
    df['trades_in_last_30_min'] * 0.3 +
    df['lot_size_vs_baseline'] * 0.3
)


df['revenge_trading_probability'] = (
    (
        df['consecutive_losses'] +
        df['trades_in_last_30_min']
    ) / 10
).clip(0, 1)

df['tilt_detection_score'] = (
    df['consecutive_losses'] * 0.5 +
    df['lot_size_vs_baseline'] * 0.3 +
    df['trades_in_last_30_min'] * 0.2
)


df['pattern_deviation_from_baseline'] = (
    abs(df['lot_size'] - baseline_lot)
)


In [13]:
df['emotional_state_estimation'] = (
    (
        df['revenge_trading_probability'] +
        df['tilt_detection_score']
    ) / 2
)


df['risk_escalation_velocity'] = (
    df['lot_size'].diff(periods=5)
)


df = df.fillna(0)




In [14]:
feature_columns = [

    
    'time_since_last_trade',
    'hour_of_day',
    'day_of_week',
    'session',
    'trade_duration_seconds',

    
    'lot_size_vs_baseline',
    'lot_size_change_from_last',
    'position_size_percentile',
    'risk_percent_of_balance',

    
    'consecutive_losses',
    'trades_in_last_30_min',
    'trades_in_session',
    'same_symbol_consecutive',

   
    'pnl_last_trade',
    'cumulative_pnl_session',
    'pnl_volatility',
    'drawdown_from_peak',

    
    'sl_present',
    'risk_reward_ratio',
    'emotional_momentum_score',

    
    'revenge_trading_probability',
    'tilt_detection_score',
    'pattern_deviation_from_baseline',
    'emotional_state_estimation',
    'risk_escalation_velocity'
]




In [15]:
scaler = RobustScaler()

df[feature_columns] = scaler.fit_transform(
    df[feature_columns]
)


feature_vectors = df[feature_columns].to_numpy()


df.to_csv(
    "behavioral_feature_dataset.csv",
    index=False
)



In [16]:

np.save(
    "behavioral_feature_vectors.npy",
    feature_vectors
)


In [17]:
pd.DataFrame(
    feature_vectors,
    columns=feature_columns
).to_csv(
    "behavioral_feature_vectors.csv",
    index=False
)


In [19]:
print("\n Feature Engineering Completed")

print("\nFeature Vector Shape:")
print(feature_vectors.shape)

print("\nSample Feature Vector:")
print(feature_vectors[0])

print("\nFeature Names:")
print(feature_columns)

print("\nProcessed Dataset Preview:")
print(df.head())


 Feature Engineering Completed

Feature Vector Shape:
(100, 25)

Sample Feature Vector:
[ 0.37735849 -0.4        -0.5         0.          3.5        -0.5
 -0.16666667 -0.5        -0.01481897  0.         -0.5        -1.
  0.         -0.43835616 -0.67092652 -0.68437571 -0.16410256  0.
  0.         -0.56058394 -0.33333333 -0.52571429  0.         -0.63888889
  0.        ]

Feature Names:
['time_since_last_trade', 'hour_of_day', 'day_of_week', 'session', 'trade_duration_seconds', 'lot_size_vs_baseline', 'lot_size_change_from_last', 'position_size_percentile', 'risk_percent_of_balance', 'consecutive_losses', 'trades_in_last_30_min', 'trades_in_session', 'same_symbol_consecutive', 'pnl_last_trade', 'cumulative_pnl_session', 'pnl_volatility', 'drawdown_from_peak', 'sl_present', 'risk_reward_ratio', 'emotional_momentum_score', 'revenge_trading_probability', 'tilt_detection_score', 'pattern_deviation_from_baseline', 'emotional_state_estimation', 'risk_escalation_velocity']

Processed Dataset Pr